# Fama-French regression: a controlled experiment

This notebook first uses simulated monthly data so the true factor exposures are known. Before inspecting the regression table, predict whether the estimated intercept should be near the simulated alpha. Then compare FF3 with FF5. The simulation is a teaching experiment, not evidence about the real stock universe.

For the data source, units, HAC errors, and limitations, see [`docs/fama-french.md`](../docs/fama-french.md).

## 1. Generate a reproducible synthetic sample

All simulated values below are illustrative. The portfolio return is constructed as RF plus a known alpha, known factor exposures, and noise.

In [ ]:
import numpy as np
import pandas as pd

from quant_models.factors.regression import run_factor_regression

random = np.random.default_rng(42)
dates = pd.date_range("2010-01-31", periods=180, freq="ME")
factor_names = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
factors = pd.DataFrame(
    random.normal(0.0, 0.03, (len(dates), len(factor_names))),
    index=dates,
    columns=factor_names,
)
factors["RF"] = random.normal(0.001, 0.0002, len(dates))
true_alpha = 0.005
true_betas = pd.Series(
    {"Mkt-RF": 1.1, "SMB": 0.2, "HML": -0.1, "RMW": 0.3, "CMA": -0.2}
)
noise = random.normal(0.0, 0.004, len(dates))
excess_returns = true_alpha + factors[true_betas.index].dot(true_betas) + noise
portfolio_returns = (excess_returns + factors["RF"]).rename("portfolio_return")
true_alpha

## 2. Compare FF3 and FF5

The dependent variable is portfolio return minus RF. Factor values are already excess returns where the column name says `Mkt-RF`. HAC errors change uncertainty estimates, not the fitted coefficients.

In [ ]:
ff3 = run_factor_regression(portfolio_returns, factors, "ff3", hac_lags=3)
ff5 = run_factor_regression(portfolio_returns, factors, "ff5", hac_lags=3)
ff3.coefficients, ff5.coefficients

## 3. Questions to answer

1. Did each regression recover an alpha near the known simulated value?
2. Which factors were deliberately omitted from FF3, and where might their exposure appear instead?
3. Why is the p-value not the probability that the portfolio has skill?
4. Why does this synthetic result say nothing about the survivorship-biased stock list?